In [ ]:
import wfdb     # reads MIT-BIH .dat/.hea/.atr files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import os

In [ ]:
data = 'mit-bih-arrhythmia-database'
# .hea files exist for every record, so use those to get unique IDs
all_files = os.listdir(data)
record_ids = sorted(set(f.split('.')[0] for f in all_files if f.endswith('.hea')))

print(f"Found {len(record_ids)} records:")
print(record_ids)

Found 48 records:
['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234']


In [3]:
from features import bandpass_filter, extract_features_from_record

In [ ]:
data_folder = data

all_dfs = []

for rec_id in record_ids:
    try:
        df,_,_ = extract_features_from_record(rec_id, data_folder=data_folder)
        all_dfs.append(df)
    except Exception as e:
        print(f"Skipped {rec_id}: {e}")

full_dataset = pd.concat(all_dfs, ignore_index=True)
print(f"Total beats across all records: {full_dataset.shape}")
print(full_dataset['true_label'].value_counts())

Total beats across all records: (112566, 7)
true_label
Normal      75052
Abnormal    37514
Name: count, dtype: int64


In [ ]:
X = full_dataset[['rr_interval', 'heart_rate', 'hrv', 'amplitude', 'qrs_width']]
y = full_dataset['true_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Models that are scale-sensitive (SVM, KNN, Logistic Regression) get wrapped
# in a Pipeline with StandardScaler. Tree-based models (RF, Decision Tree) and
# Naive Bayes don't need scaling, but a pipeline keeps everything uniform.
models = {
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    'Naive Bayes': Pipeline([
        ('clf', GaussianNB())
    ]),
    'SVM (RBF kernel)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', class_weight='balanced', random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
}

results = []
trained_models = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    # macro avg treats both classes equally, so it's a fairer comparison metric
    # than accuracy given the Normal/Abnormal class imbalance
    prec = precision_score(y_test, y_pred, pos_label='Abnormal')
    rec = recall_score(y_test, y_pred, pos_label='Abnormal')
    f1 = f1_score(y_test, y_pred, pos_label='Abnormal')
    f1_macro = f1_score(y_test, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Abnormal Precision': prec,
        'Abnormal Recall': rec,
        'Abnormal F1': f1,
        'Macro F1': f1_macro,
    })
    trained_models[name] = pipe

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values('Macro F1', ascending=False).reset_index(drop=True)
print("\n\n=== Model Comparison (sorted by Macro F1) ===")
print(results_df.to_string(index=False))

Accuracy: 0.9488318379674869

Confusion Matrix:
[[ 6974   529]
 [  623 14388]]

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.92      0.93      0.92      7503
      Normal       0.96      0.96      0.96     15011

    accuracy                           0.95     22514
   macro avg       0.94      0.94      0.94     22514
weighted avg       0.95      0.95      0.95     22514



In [ ]:
# Pick the best model by Macro F1 (fairer than raw accuracy given class imbalance;
# recall on 'Abnormal' matters most clinically, so cross-check that column too
# before finalizing, in case a close second has meaningfully better recall).
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

joblib.dump(best_model, 'ecg_model.pkl')
print(f"Best model: {best_model_name} (Macro F1 = {results_df.iloc[0]['Macro F1']:.4f})")
print("Saved as ecg_model.pkl")

Model saved.
